# Nemotron Reasoning — Baseline SFT Notebook w/ Unsloth

**Strategy:** Supervised Fine-Tuning (SFT) on the full training set using LoRA

## 1. Setup

In [1]:
!pip install -q --no-index --find-links /kaggle/input/datasets/mayukh18/nemotron-packages/packages unsloth trl peft transformers datasets accelerate bitsandbytes 
!pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
!pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.


## Config

In [4]:
!rm -rf /kaggle/tmp/*

In [ ]:

from unsloth import FastLanguageModel


In [ ]:

import os
import re
import math
import statistics
from pathlib import Path
import zipfile
import time
import pandas as pd
from datasets import Dataset
import kagglehub
import torch


In [ ]:

DATA_DIR    = Path("/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge")
TRAIN_PATH  = DATA_DIR / "train.csv"   # used only to build the validation split
MODEL_PATH  = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")

FORMATTED_TRAIN_CSV = "/kaggle/input/datasets/zuhairsan/new-cot-085/formatted_train_dataset_backup.csv"

ADAPTER_DIR           = "nemotron-lora-adapter"
CHECKPOINT_OUTPUT_DIR = "/kaggle/tmp/checkpoints"
SUBMISSION_ZIP        = "submission.zip"

# LoRA config
LORA_RANK    = 32
LORA_ALPHA   = 16
LORA_DROPOUT = 0.1

# Training config
MAX_SEQ_LEN  = 3500
NUM_EPOCHS   = 2
BATCH_SIZE   = 2
GRAD_ACCUM   = 1
LR           = 1e-4
WARMUP_RATIO = 0.03

# Eval & loss
EVAL_SAVE_STEPS   = 300
BOXED_LOSS_WEIGHT = 5.0  # upweight final \boxed{} tokens (1.0 = disabled)

print("Config ready.")
print(f"  model      : {MODEL_PATH}")
print(f"  train CSV  : {FORMATTED_TRAIN_CSV}")
print(f"  LoRA       : rank={LORA_RANK}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"  training   : epochs={NUM_EPOCHS}, batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM}, lr={LR}, warmup={WARMUP_RATIO}")
print(f"  seq_len    : {MAX_SEQ_LEN}, eval_steps={EVAL_SAVE_STEPS}, boxed_loss_weight={BOXED_LOSS_WEIGHT}")


## 2. Load & Inspect Data

In [ ]:

train_df = pd.read_csv(TRAIN_PATH)
print(f"Train: {len(train_df):,} rows — columns: {list(train_df.columns)}")


In [9]:
def classify_equation_vs_symbol(prompt: str) -> str:
    q = prompt.strip()

    # extract the target expression after "determine the result for" if present
    m = re.search(r"determine the result for[:\s]*([^\n\r]*)", q, re.IGNORECASE)
    expr = m.group(1).strip() if m else q

    # counts
    digit_count = len(re.findall(r"\d", expr))
    alpha_count = len(re.findall(r"[A-Za-z]", expr))
    symbol_count = len(re.findall(r"[^\w\s]", expr))  # punctuation / symbols

    # rule 1: numeric_equation if expression contains two numeric groups separated by a non-digit operator
    if re.search(r"\d+\s*[^0-9\s]\s*\d+", expr):
        return "numeric_equation"

    # rule 2: symbol_transform if symbols dominate and digits/letters are scarce
    total_chars = max(1, len(expr))
    if symbol_count / total_chars > 0.5 and digit_count + alpha_count < max(2, symbol_count//2):
        return "symbol_transform"

    # fallback: look at examples in the prompt body (presence of lines with digits → numeric)
    if re.search(r"^\s*\d+[^=\n]*=", q, re.MULTILINE):
        return "numeric_equation"
    if re.search(r"[^\w\s]{2,}", q):  # repeated punctuation sequences
        return "symbol_transform"

    return "unknown"

In [ ]:

# Build validation_df as a small random holdout from train.csv (5% per category, max 50 per category)
import random
random.seed(42)

VAL_FRACTION = 0.05
VAL_MAX_PER_CATEGORY = 50

def _classify(prompt):
    p = prompt.lower()
    if "bit manipulation" in p:        return "bit_manipulation"
    if "secret encryption rules" in p: return "text_decryption"
    if "unit conversion" in p:         return "unit_conversion"
    if "numeral system" in p:          return "numeral_system"
    if "gravitational constant" in p:  return "gravity_physics"
    if "transformation rules" in p:    return "symbol_or_numeric"
    return "unknown"

train_df["category"] = train_df["prompt"].apply(_classify)

val_rows = []
for cat, group in train_df.groupby("category"):
    n = min(max(1, int(len(group) * VAL_FRACTION)), VAL_MAX_PER_CATEGORY)
    val_rows.append(group.sample(n, random_state=42))

validation_df = pd.concat(val_rows).reset_index(drop=True)[["id", "prompt", "answer", "category"]]

print(f"Validation samples: {len(validation_df):,}")
for cat, cnt in validation_df["category"].value_counts().sort_index().items():
    print(f"  {cat}: {cnt}")


## Finalize

In [ ]:

BOXED_INSTRUCTION = (
    "\nPlease put your final answer inside `\\boxed{}`. "
    "For example: `\\boxed{your answer}`"
)

def format_train_row(prompt: str, answer: str) -> dict:
    """Minimal formatter used to build the validation set."""
    return {
        "messages": [
            {"role": "user",      "content": prompt + BOXED_INSTRUCTION},
            {"role": "assistant", "content": f"\\boxed{{{answer}}}"},
        ]
    }

# ── Load training data from pre-formatted CSV ──
_fmt_df = pd.read_csv(FORMATTED_TRAIN_CSV)
print(f"Loaded {len(_fmt_df):,} rows from {FORMATTED_TRAIN_CSV}")
print(f"Columns: {list(_fmt_df.columns)}")

train_dataset = Dataset.from_list([
    {
        "messages": [
            {"role": "user",      "content": str(row["user_content"])},
            {"role": "assistant", "content": str(row["assistant_content"])},
        ]
    }
    for _, row in _fmt_df.iterrows()
])

print(f"Training examples: {len(train_dataset):,}")
if "type" in _fmt_df.columns:
    print("Type distribution:")
    for t, cnt in _fmt_df["type"].value_counts().sort_index().items():
        print(f"  {t}: {cnt}")


In [ ]:

# Training data is now loaded from formatted_train_dataset_backup.csv — no need to re-save here.
# (original save code commented out to avoid overwriting the source CSV)

# output_csv = []
# for record in train_records:
#     user_content = record["messages"][0]["content"]
#     assistant_content = record["messages"][1]["content"]
#     output_csv.append({"user_content": user_content, "assistant_content": assistant_content})
# pd.DataFrame(output_csv).to_csv("formatted_train_dataset.csv", index=False)


In [22]:
# compute prompt lengths and show stats
train_dataset = train_dataset.map(lambda x: {"prompt_len": len(x["messages"][0]["content"])})
arr = train_dataset["prompt_len"]
import numpy as np
print(f"count={len(arr)}, mean={np.mean(arr):.1f}, std={np.std(arr):.1f}, min={np.min(arr)}, 25%={np.percentile(arr,25)}, 50%={np.median(arr)}, 75%={np.percentile(arr,75)},98%={np.percentile(arr,98)}, max={np.max(arr)}")

Map:   0%|          | 0/3910 [00:00<?, ? examples/s]

count=3910, mean=446.6, std=111.7, min=261, 25%=328.0, 50%=460.0, 75%=551.0,98%=593.0, max=593


In [23]:
# answer length stats
train_dataset = train_dataset.map(lambda x: {"answer_len": len(x["messages"][1]["content"])})
arr = train_dataset["answer_len"]
print(f"count={len(arr)}, mean={np.mean(arr):.1f}, std={np.std(arr):.1f}, min={np.min(arr)}, 25%={np.percentile(arr,25)}, 50%={np.median(arr)}, 75%={np.percentile(arr,75)},95%={np.percentile(arr,95)},96%={np.percentile(arr,96)},97%={np.percentile(arr,97)},98%={np.percentile(arr,98)},99%={np.percentile(arr,99)}, max={np.max(arr)}")

Map:   0%|          | 0/3910 [00:00<?, ? examples/s]

count=3910, mean=3485.3, std=1768.7, min=550, 25%=2579.25, 50%=3303.0, 75%=4316.75,95%=6871.099999999999,96%=7035.28,97%=7266.22,98%=7491.279999999999,99%=7785.659999999996, max=8724


In [24]:
# #set MAX_SEQ_LEN to 99th percentile of prompt_len + answer_len to cover most examples without truncation
# train_dataset = train_dataset.map(lambda x: {"total_len": x["prompt_len"] + x["answer_len"]})
# arr = train_dataset["total_len"]
# print(f"Total length (prompt + answer) stats: count={len(arr)}, mean={np.mean(arr):.1f}, std={np.std(arr):.1f}, min={np.min(arr)}, 25%={np.percentile(arr,25)}, 50%={np.median(arr)}, 75%={np.percentile(arr,75)}, 99%={np.percentile(arr,99)}, max={np.max(arr)}")
# MAX_SEQ_LEN = int(np.percentile(arr, 75)) + 256  # add some buffer for special tokens
# print(f"Setting MAX_SEQ_LEN to {MAX_SEQ_LEN} to cover 99% of examples without truncation.")

## 4. Load Base Model with LoRA (4-bit via Unsloth)

In [25]:
if 'model' in globals() and 'tokenizer' in globals():
    del model, tokenizer

    import gc
    gc.collect()
    torch.cuda.empty_cache()

In [26]:

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = MAX_SEQ_LEN, # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    trust_remote_code = True,
    unsloth_force_compile = False,
    attn_implementation = "eager",
    dtype = torch.bfloat16,  # explicit bf16; RTX PRO 6000 Blackwell supports bf16
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.3.17: Fast Nemotron_H patching. Transformers: 4.57.6. vLLM: 0.18.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards:   0%|          | 0/13 [00:00<?, ?it/s]

/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1 does not have a padding token! Will use pad_token = <SPECIAL_999>.


In [27]:
target_modules=[
    'out_proj', 'v_proj', 'q_proj', 'down_proj', 'embed_tokens',
    'k_proj', 'in_proj', 'up_proj', 'o_proj', 'lm_head', 'gate_proj'
]

In [ ]:

model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_RANK,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    target_modules             = target_modules,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
)
model.print_trainable_parameters()


## 5. Train with SFTTrainer

In [ ]:
!pip install psutil

In [30]:
# pip install psutil
import psutil
mem = psutil.virtual_memory()
print(f"RAM total: {mem.total/1024**3:.2f} GB, used: {mem.used/1024**3:.2f} GB ({mem.percent}%)")

RAM total: 176.88 GB, used: 4.85 GB (3.7%)


In [ ]:

from trl import SFTTrainer
from transformers import TrainingArguments

train_dataset = train_dataset.map(
    lambda ex: {
        "text": tokenizer.apply_chat_template(
            ex["messages"],
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=True,
        )
    }
)


In [ ]:

from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback
import inspect

bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16_ok = torch.cuda.is_available() and not bf16_ok
steps_per_epoch = max(1, len(train_dataset) // max(1, BATCH_SIZE * GRAD_ACCUM))
warmup_steps = max(10, int(steps_per_epoch * max(1, NUM_EPOCHS) * WARMUP_RATIO))

# Build validation set (plain \boxed{answer} targets — no trace needed)
val_records = [
    format_train_row(str(row["prompt"]), str(row["answer"]))
    for _, row in validation_df.iterrows()
]
val_dataset = Dataset.from_list(val_records).map(
    lambda ex: {
        "text": tokenizer.apply_chat_template(
            ex["messages"],
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=True,
        )
    }
)

# ── Weighted loss: upweight final \boxed{answer} tokens ──────────────────
_compute_loss_fn = None
if BOXED_LOSS_WEIGHT > 1.0:
    _boxed_marker_ids = tokenizer.encode("\\boxed{", add_special_tokens=False)
    _boxed_weight = float(BOXED_LOSS_WEIGHT)
    print(f"[loss] \\boxed{{ marker tokenizes to {len(_boxed_marker_ids)} token IDs: {_boxed_marker_ids}")

    def _weighted_boxed_loss(outputs, labels, num_items_in_batch=None):
        logits = outputs.logits if hasattr(outputs, 'logits') else outputs[0]
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        batch_size, seq_len = shift_labels.shape

        loss_fct = torch.nn.CrossEntropyLoss(reduction='none', ignore_index=-100)
        per_token_loss = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1)
        ).view(batch_size, seq_len)

        weights = torch.ones(batch_size, seq_len, device=per_token_loss.device)
        marker = torch.tensor(_boxed_marker_ids, device=shift_labels.device)
        marker_len = len(_boxed_marker_ids)

        for bi in range(batch_size):
            last_pos = -1
            for i in range(seq_len - marker_len + 1):
                if torch.equal(shift_labels[bi, i:i+marker_len], marker):
                    last_pos = i
            if last_pos >= 0:
                weights[bi, last_pos:] = _boxed_weight

        mask = (shift_labels != -100).float()
        weighted_loss = (per_token_loss * weights * mask).sum() / (weights * mask).sum()
        return weighted_loss

    _compute_loss_fn = _weighted_boxed_loss
    print(f"[loss] Weighted loss ready: final \\boxed{{}} region gets {_boxed_weight}x weight")
else:
    print("[loss] BOXED_LOSS_WEIGHT <= 1.0 — using default uniform loss")

args = TrainingArguments(
    output_dir=CHECKPOINT_OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=14,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    bf16=bf16_ok,
    fp16=fp16_ok,
    logging_steps=EVAL_SAVE_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_SAVE_STEPS,
    save_strategy="no",
    save_steps=EVAL_SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=False,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    optim="adamw_8bit",
    gradient_checkpointing=False,
    seed=42,
    report_to="none",
)

_extra_kwargs = {}
if _compute_loss_fn is not None:
    if 'compute_loss_func' in inspect.signature(SFTTrainer.__init__).parameters:
        _extra_kwargs['compute_loss_func'] = _compute_loss_fn
        print("[loss] Passing compute_loss_func to SFTTrainer")
    else:
        print("[loss] compute_loss_func not supported in this trl version — skipping weighted loss")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=args,
    packing=False,
    **_extra_kwargs,
)


In [33]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  63643 MiB |  63651 MiB | 124914 MiB |  61271 MiB |
|       from large pool |  60298 MiB |  60306 MiB | 119867 MiB |  59569 MiB |
|       from small pool |   3344 MiB |   3345 MiB |   5047 MiB |   1702 MiB |
|---------------------------------------------------------------------------|
| Active memory         |  63643 MiB |  63651 MiB | 124914 MiB |  61271 MiB |
|       from large pool |  60298 MiB |  60306 MiB | 119867 MiB |

In [ ]:

print("Starting training...")
train_out = trainer.train()
print("Training complete.")
print({k: train_out.metrics.get(k) for k in ["train_runtime", "train_samples_per_second", "train_steps_per_second", "train_loss"]})


## 6. Save LoRA Adapter

In [ ]:

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

assert os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")), \
    "adapter_config.json missing!"

print(f"Adapter saved to ./{ADAPTER_DIR}/")
print("Files:", os.listdir(ADAPTER_DIR))


## 7. Submission

The competition expects a zip archive containing the LoRA adapter directory (with `adapter_config.json` at the root of the archive or a sub-directory).


In [ ]:

with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(ADAPTER_DIR):
        zf.write(os.path.join(ADAPTER_DIR, fname), arcname=os.path.join(ADAPTER_DIR, fname))

with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zf:
    names = zf.namelist()

has_config = any("adapter_config.json" in n for n in names)
print(f"Files in {SUBMISSION_ZIP}: {names}")
print(f"adapter_config.json present: {has_config}")
